# 🎓 NuevaMente — Hackathon ONE G10
### Sistema Inteligente de Adaptación y Generación de Contenido Educativo

**Oracle Next Education & Alura**

Este notebook implementa un MVP orientado a:

- Ingestión de documentos **PDF, Markdown y TXT**.
- Extracción, limpieza y segmentación (*chunking*).
- Generación de **embeddings** y búsqueda vectorial con **FAISS**.
- Pipeline **RAG** para recuperar evidencia del documento.
- Adaptación mediante **Google Gemini** según perfil, formato pedagógico, nicho y nivel.
- Revisión de fidelidad y salida estructurada en **JSON**.
- Interfaz interactiva con **Gradio**.
- Integración opcional durante desarrollo y obligatoria para la entrega con **OCI Object Storage**.
- Casos de prueba para la demostración del hackathon.

> Ejecutá las celdas en orden. Las credenciales se leen desde **Colab Secrets** y no deben escribirse directamente en el notebook.

## 1. Instalación de dependencias

In [1]:
!pip -q install -U pypdf sentence-transformers faiss-cpu google-genai gradio pydantic oci
print("✅ Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.9/36.9 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

## 2. Importaciones y carpetas del proyecto

In [2]:
import os, re, json, tempfile, uuid, datetime, hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import faiss
import gradio as gr
from pypdf import PdfReader
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

BASE_DIR = Path("/content/NuevaMente")
DOCS_DIR = BASE_DIR / "documentos"
RESULTS_DIR = BASE_DIR / "resultados"
for p in (BASE_DIR, DOCS_DIR, RESULTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("✅ Entorno listo:", BASE_DIR)

✅ Entorno listo: /content/NuevaMente


## 3. Configuración segura de Gemini

En Colab:
1. Abrí **Secrets** (ícono de llave).
2. Creá un secreto llamado `GEMINI_API_KEY`.
3. Pegá allí tu API key y habilitá el acceso al notebook.

In [28]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

# Modelo configurable: si fuera necesario, se puede cambiar aquí.
GEMINI_MODEL = "gemini-3.6-flash"

print("✅ Gemini configurado sin exponer la API key.")

✅ Gemini configurado sin exponer la API key.


## 4. Parámetros funcionales de NuevaMente

In [6]:
PERFILES = [
    "Principiante / Transición de Carrera",
    "Desarrollador Junior / Semi Senior",
    "Líder Técnico / Arquitecto",
    "Gestor / Ejecutivo (No Técnico)"
]

FORMATOS = [
    "Guía Práctica Paso a Paso (Tutorial)",
    "Flashcards de Memorización",
    "Quiz Interactivo con Justificaciones",
    "Resumen Ejecutivo (TL;DR)",
    "Guion de Clase / Video"
]

NICHOS = ["General", "Fintech", "Salud", "E-commerce"]
NIVELES = ["Básico", "Didáctico", "Intermedio", "Avanzado"]

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
TOP_K = 6

print("✅ Parámetros cargados.")

✅ Parámetros cargados.


## 5. Ingestión: PDF, Markdown y TXT

In [34]:
def extraer_texto(ruta: str) -> str:
    ruta = Path(ruta)
    ext = ruta.suffix.lower()

    if ext == ".pdf":
        reader = PdfReader(str(ruta))
        paginas = []

        for i, page in enumerate(reader.pages, start=1):
            # "layout" suele conservar mejor palabras y espacios
            try:
                texto = page.extract_text(
                    extraction_mode="layout"
                ) or ""
            except TypeError:
                # Compatibilidad con versiones anteriores de pypdf
                texto = page.extract_text() or ""

            texto = texto.replace("\x00", "")
            texto = texto.strip()

            if texto:
                paginas.append(
                    f"\n[PÁGINA {i}]\n{texto}"
                )

        texto_final = "\n".join(paginas)

        if len(texto_final.strip()) < 50:
            raise ValueError(
                "No se pudo extraer suficiente texto del PDF."
            )

        return texto_final

    if ext in {".md", ".markdown", ".txt"}:
        return ruta.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    raise ValueError(
        "Formato no soportado. Utilizá PDF, Markdown o TXT."
    )


def limpiar_texto(texto: str) -> str:
    texto = texto.replace("\x00", " ")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


print("✅ Lectores preparados.")

✅ Lectores preparados.


## 6. Chunking

In [8]:
def crear_chunks(texto: str, chunk_size: int = CHUNK_SIZE,
                 overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not texto:
        return []

    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = min(inicio + chunk_size, len(texto))
        fragmento = texto[inicio:fin].strip()
        if fragmento:
            chunks.append(fragmento)
        if fin == len(texto):
            break
        inicio = max(fin - overlap, inicio + 1)

    return chunks

print("✅ Función de chunking preparada.")

✅ Función de chunking preparada.


## 7. Embeddings + FAISS

Se utiliza un modelo de embeddings local de `sentence-transformers`, por lo que los embeddings no requieren una segunda API externa.

In [9]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

class VectorRAG:
    def __init__(self):
        self.chunks = []
        self.index = None

    def indexar(self, chunks: List[str]):
        if not chunks:
            raise ValueError("No hay fragmentos para indexar.")

        self.chunks = chunks
        vectors = embedding_model.encode(
            chunks, normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")

        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)
        return len(chunks)

    def buscar(self, consulta: str, k: int = TOP_K) -> List[Dict[str, Any]]:
        if self.index is None:
            raise ValueError("El Vector Store todavía no fue creado.")

        q = embedding_model.encode(
            [consulta], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")

        k = min(k, len(self.chunks))
        scores, ids = self.index.search(q, k)

        return [
            {"chunk_id": int(idx), "score": float(score), "texto": self.chunks[idx]}
            for score, idx in zip(scores[0], ids[0]) if idx >= 0
        ]

rag = VectorRAG()
print("✅ Modelo de embeddings y FAISS preparados.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo de embeddings y FAISS preparados.


## 8. Esquema JSON esperado

In [10]:
class Metadatos(BaseModel):
    perfil_aplicado: str
    formato_generado: str
    tiempo_estimado_estudio_minutos: int
    conceptos_clave: List[str]
    prerrequisitos: List[str] = []

class EvaluacionCalidad(BaseModel):
    fidelidad_fuente: str
    claridad_pedagogica: str
    observaciones: str

class ResultadoEducativo(BaseModel):
    status: str = "exito"
    metadatos: Metadatos
    contenido_adaptado: Dict[str, Any]
    evaluacion_calidad: EvaluacionCalidad
    fuentes_recuperadas: List[int] = []

print("✅ Esquema Pydantic preparado.")

✅ Esquema Pydantic preparado.


## 9. Utilidades para obtener JSON válido desde Gemini

In [11]:
def extraer_json_respuesta(texto: str) -> Dict[str, Any]:
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?\\s*", "", texto, flags=re.I)
    texto = re.sub(r"\\s*```$", "", texto)
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        ini = texto.find("{")
        fin = texto.rfind("}")
        if ini >= 0 and fin > ini:
            return json.loads(texto[ini:fin+1])
        raise

def llamar_gemini(prompt: str) -> str:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )
    return response.text

print("✅ Utilidades de Gemini preparadas.")

✅ Utilidades de Gemini preparadas.


## 10. Motor RAG + adaptación pedagógica

El LLM recibe únicamente los fragmentos recuperados como evidencia principal.  
El prompt le indica que no invente hechos ausentes en la fuente.

In [12]:
def construir_consulta(perfil, formato, nicho, nivel):
    return (
        f"Conceptos técnicos, definiciones, procedimientos, requisitos y ejemplos "
        f"necesarios para enseñar este documento a un perfil {perfil}, "
        f"en formato {formato}, contexto {nicho}, nivel {nivel}."
    )

def generar_contenido(perfil: str, formato: str, nicho: str, nivel: str):
    consulta = construir_consulta(perfil, formato, nicho, nivel)
    recuperados = rag.buscar(consulta, TOP_K)

    contexto = "\\n\\n".join(
        f"[FUENTE {r['chunk_id']}]\\n{r['texto']}" for r in recuperados
    )

    prompt = f"""
Eres el motor pedagógico de NuevaMente.

Tu tarea es transformar EXCLUSIVAMENTE la evidencia recuperada del documento
en material educativo. No inventes características, cifras, pasos, requisitos
ni definiciones que no estén respaldados por la evidencia.

PARÁMETROS:
- Perfil destinatario: {perfil}
- Formato pedagógico: {formato}
- Nicho/contexto: {nicho}
- Nivel de detalle: {nivel}

EVIDENCIA RAG:
{contexto}

Devuelve ÚNICAMENTE JSON válido con esta estructura:
{{
  "status": "exito",
  "metadatos": {{
    "perfil_aplicado": "{perfil}",
    "formato_generado": "{formato}",
    "tiempo_estimado_estudio_minutos": 10,
    "conceptos_clave": ["..."],
    "prerrequisitos": ["..."]
  }},
  "contenido_adaptado": {{
    "titulo": "...",
    "introduccion_contextualizada": "...",
    "items": []
  }},
  "evaluacion_calidad": {{
    "fidelidad_fuente": "Alta/Media/Baja",
    "claridad_pedagogica": "Alta/Media/Baja",
    "observaciones": "..."
  }},
  "fuentes_recuperadas": {[r['chunk_id'] for r in recuperados]}
}}

Adapta la estructura interna de "items" al formato solicitado:
- Flashcards: frente, dorso, pista_didactica.
- Quiz: pregunta, opciones, respuesta_correcta, justificacion.
- Tutorial: paso, titulo, explicacion.
- Resumen ejecutivo: punto, explicacion.
- Guion: seccion, narracion.

No uses Markdown alrededor del JSON.
"""

    bruto = llamar_gemini(prompt)
    data = extraer_json_respuesta(bruto)
    validado = ResultadoEducativo.model_validate(data)
    return validado.model_dump(), recuperados

print("✅ Motor pedagógico RAG preparado.")

✅ Motor pedagógico RAG preparado.


## 11. Revisión de fidelidad

Esta segunda llamada actúa como revisor: compara el resultado generado con la evidencia recuperada y señala posibles afirmaciones sin respaldo.

In [13]:
def revisar_fidelidad(resultado: Dict[str, Any],
                       recuperados: List[Dict[str, Any]]) -> Dict[str, Any]:
    evidencia = "\\n\\n".join(
        f"[FUENTE {r['chunk_id']}] {r['texto']}" for r in recuperados
    )

    prompt = f"""
Actúa como revisor de fidelidad de un sistema RAG.

EVIDENCIA:
{evidencia}

CONTENIDO GENERADO:
{json.dumps(resultado, ensure_ascii=False)}

Evalúa si las afirmaciones técnicas están respaldadas por la evidencia.
Devuelve ÚNICAMENTE JSON válido:
{{
  "veredicto": "APROBADO" o "REVISAR",
  "fidelidad_estimada": número entre 0 y 1,
  "hallazgos": ["..."],
  "recomendacion": "..."
}}

No inventes evidencia.
"""

    revision = extraer_json_respuesta(llamar_gemini(prompt))
    return revision

print("✅ Revisor de fidelidad preparado.")

✅ Revisor de fidelidad preparado.


## 12. Procesamiento completo de un documento

In [14]:
ESTADO = {
    "archivo": None,
    "texto": None,
    "chunks": [],
}

def preparar_documento(ruta: str):
    texto = limpiar_texto(extraer_texto(ruta))
    chunks = crear_chunks(texto)

    if not chunks:
        raise ValueError("No fue posible extraer contenido textual del documento.")

    cantidad = rag.indexar(chunks)

    ESTADO["archivo"] = ruta
    ESTADO["texto"] = texto
    ESTADO["chunks"] = chunks

    return {
        "archivo": Path(ruta).name,
        "caracteres_extraidos": len(texto),
        "chunks_indexados": cantidad
    }

print("✅ Pipeline de preparación listo.")

✅ Pipeline de preparación listo.


## 13. OCI Object Storage

Para la entrega, el documento original y el JSON generado deben persistirse en OCI Object Storage.

Configurá estos Secrets de Colab cuando tengas tu cuenta/bucket preparados:

- `OCI_USER_OCID`
- `OCI_TENANCY_OCID`
- `OCI_FINGERPRINT`
- `OCI_REGION`
- `OCI_NAMESPACE`
- `OCI_BUCKET`
- `OCI_PRIVATE_KEY`

`OCI_PRIVATE_KEY` debe contener el contenido de la clave privada PEM.

La aplicación seguirá funcionando localmente en Colab si OCI todavía no está configurado, pero **antes de la entrega debe probarse con OCI activo**.

In [18]:
import oci
from google.colab import userdata

def secret_opcional(nombre):
    try:
        return userdata.get(nombre)
    except Exception:
        return None

def obtener_oci():
    valores = {
        "user": secret_opcional("OCI_USER_OCID"),
        "tenancy": secret_opcional("OCI_TENANCY_OCID"),
        "fingerprint": secret_opcional("OCI_FINGERPRINT"),
        "region": secret_opcional("OCI_REGION"),
        "key_content": secret_opcional("OCI_PRIVATE_KEY"),
    }
    namespace = secret_opcional("OCI_NAMESPACE")
    bucket = secret_opcional("OCI_BUCKET")

    if not all(valores.values()) or not namespace or not bucket:
        return None, None, None

    config = {
        "user": valores["user"],
        "tenancy": valores["tenancy"],
        "fingerprint": valores["fingerprint"],
        "region": valores["region"],
        "key_content": valores["key_content"],
    }
    oci.config.validate_config(config)
    return oci.object_storage.ObjectStorageClient(config), namespace, bucket

def subir_bytes_oci(nombre_objeto: str, contenido: bytes):
    cliente, namespace, bucket = obtener_oci()
    if cliente is None:
        return {
            "configurado": False,
            "status": "OCI no configurado todavía",
            "objeto": None
        }

    cliente.put_object(namespace, bucket, nombre_objeto, contenido)
    return {
        "configurado": True,
        "status": "completado",
        "bucket": bucket,
        "objeto": nombre_objeto
    }

print("✅ Módulo OCI preparado.")

cliente, namespace, bucket = obtener_oci()

if cliente is None:
    print("❌ OCI no está configurado. Revisá los Secrets.")
else:
    try:
        respuesta = cliente.list_objects(
            namespace_name=namespace,
            bucket_name=bucket,
            limit=10
        )

        print("✅ CONEXIÓN OCI CORRECTA")
        print("Namespace:", namespace)
        print("Bucket:", bucket)
        print("Objetos actuales:", len(respuesta.data.objects))

    except Exception as e:
        print("❌ Error al conectar con OCI:")
        print(e)

✅ Módulo OCI preparado.
✅ CONEXIÓN OCI CORRECTA
Namespace: axvaymhx6tyc
Bucket: nuevamente-contenidos-educativos
Objetos actuales: 0


## 14. Función principal

In [23]:
def procesar_nuevamente(archivo, perfil, formato, nicho, nivel):
    print("📥 Archivo recibido:", archivo)

    if not archivo:
        raise gr.Error("Primero cargá un documento PDF, Markdown o TXT.")

    # Con gr.File(type="filepath"), archivo ya debe llegar como ruta
    if isinstance(archivo, str):
        ruta = archivo
    elif hasattr(archivo, "name"):
        ruta = archivo.name
    else:
        ruta = str(archivo)

    print("📄 Ruta detectada:", ruta)

    if not os.path.exists(ruta):
        raise gr.Error(f"No se encontró el archivo temporal: {ruta}")

    # 1. Preparar documento + RAG
    info = preparar_documento(ruta)
    print("✅ Documento preparado:", info)

    # 2. Subir documento original a OCI
    original_bytes = Path(ruta).read_bytes()

    oci_original = subir_bytes_oci(
        f"documentos/{Path(ruta).name}",
        original_bytes
    )

    print("☁️ OCI original:", oci_original)

    # 3. Generar contenido con RAG + Gemini
    resultado, recuperados = generar_contenido(
        perfil,
        formato,
        nicho,
        nivel
    )

    print("🤖 Contenido generado.")

    # 4. Revisar fidelidad
    revision = revisar_fidelidad(resultado, recuperados)

    resultado["revision_fidelidad"] = revision
    resultado["documento"] = info
    resultado["almacenamiento_oci_original"] = oci_original

    # 5. Crear JSON
    nombre_json = (
        f"resultado_"
        f"{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid.uuid4().hex[:6]}.json"
    )

    contenido_json = json.dumps(
        resultado,
        ensure_ascii=False,
        indent=2
    )

    ruta_json = RESULTS_DIR / nombre_json

    ruta_json.write_text(
        contenido_json,
        encoding="utf-8"
    )

    # 6. Subir resultado a OCI
    oci_resultado = subir_bytes_oci(
        f"resultados/{nombre_json}",
        contenido_json.encode("utf-8")
    )

    resultado["almacenamiento_oci_resultado"] = oci_resultado

    ruta_json.write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    # 7. Mostrar evidencia RAG
    resumen_fuentes = "\n\n".join(
        f"Chunk {r['chunk_id']} | similitud {r['score']:.3f}\n"
        f"{r['texto'][:700]}"
        for r in recuperados
    )

    print("🎉 PROCESO COMPLETO")

    return resultado, resumen_fuentes, str(ruta_json)

print("✅ Función principal corregida.")

✅ Función principal corregida.


## 15. Interfaz Gradio

La interfaz permite demostrar:
- carga de documento;
- selección de perfil;
- formato pedagógico;
- nicho;
- nivel;
- contenido JSON;
- evidencia recuperada;
- descarga del resultado.

In [38]:
with gr.Blocks(title="NuevaMente") as demo:
    gr.Markdown(
        '''
        # 🎓 NuevaMente
        ### De documentación técnica a experiencias de aprendizaje personalizadas
        Cargá un documento y elegí cómo querés aprender su contenido.
        '''
    )

    with gr.Row():
        archivo = gr.File(
            label="📄 Documento técnico",
            file_types=[".pdf", ".md", ".markdown", ".txt"],
            type="filepath"
        )
        with gr.Column():
            perfil = gr.Dropdown(PERFILES, value=PERFILES[0], label="👤 Perfil")
            formato = gr.Dropdown(FORMATOS, value=FORMATOS[1], label="📚 Formato")
            nicho = gr.Dropdown(NICHOS, value=NICHOS[0], label="🏢 Contexto")
            nivel = gr.Dropdown(NIVELES, value=NIVELES[1], label="🎯 Nivel")

    generar = gr.Button("✨ Generar contenido", variant="primary")

    with gr.Tab("📖 Resultado"):
        salida_json = gr.JSON(label="Paquete educativo")

    with gr.Tab("🔎 Evidencia RAG"):
        fuentes = gr.Textbox(label="Fragmentos recuperados", lines=18)

    descarga = gr.File(label="⬇️ Descargar JSON")

    generar.click(
        fn=procesar_nuevamente,
        inputs=[archivo, perfil, formato, nicho, nivel],
        outputs=[salida_json, fuentes, descarga]
    )

print("✅ Interfaz creada. Ejecutá la siguiente celda para abrirla.")

✅ Interfaz creada. Ejecutá la siguiente celda para abrirla.


## 16. Lanzar NuevaMente

In [ ]:
demo.launch(
    share=True,
    debug=True,
    inline=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://177df81e1c61836c02.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📥 Archivo recibido: /tmp/gradio/e21280edc4b0b274e9e54259c7b935e0410f5aba355de3993eb790da37b8dfb2/documento_prueba_nuevamente.pdf
📄 Ruta detectada: /tmp/gradio/e21280edc4b0b274e9e54259c7b935e0410f5aba355de3993eb790da37b8dfb2/documento_prueba_nuevamente.pdf
✅ Documento preparado: {'archivo': 'documento_prueba_nuevamente.pdf', 'caracteres_extraidos': 5214, 'chunks_indexados': 6}
☁️ OCI original: {'configurado': True, 'status': 'completado', 'bucket': 'nuevamente-contenidos-educativos', 'objeto': 'documentos/documento_prueba_nuevamente.pdf'}
🤖 Contenido generado.
🎉 PROCESO COMPLETO
📥 Archivo recibido: /tmp/gradio/e21280edc4b0b274e9e54259c7b935e0410f5aba355de3993eb790da37b8dfb2/documento_prueba_nuevamente.pdf
📄 Ruta detectada: /tmp/gradio/e21280edc4b0b274e9e54259c7b935e0410f5aba355de3993eb790da37b8dfb2/documento_prueba_nuevamente.pdf
✅ Documento preparado: {'archivo': 'documento_prueba_nuevamente.pdf', 'caracteres_extraidos': 5214, 'chunks_indexados': 6}
☁️ OCI original: {'configurado': Tru

# 17. Plan de pruebas para la demo

Para cumplir la demostración, utilizá **un mismo documento técnico** y ejecutá al menos estos escenarios:

### Escenario A
- Perfil: **Principiante / Transición de Carrera**
- Formato: **Flashcards**
- Nicho: **General**
- Nivel: **Didáctico**

### Escenario B
- Perfil: **Desarrollador Junior / Semi Senior**
- Formato: **Guía Práctica Paso a Paso**
- Nicho: **Fintech** o el contexto que corresponda al documento
- Nivel: **Intermedio**

### Escenario C
- Perfil: **Gestor / Ejecutivo (No Técnico)**
- Formato: **Resumen Ejecutivo (TL;DR)**
- Nicho: **General**
- Nivel: **Básico**

Durante la grabación mostrar:
1. Documento cargado.
2. Cambio de parámetros.
3. Resultado adaptado.
4. Evidencia recuperada por RAG.
5. JSON descargable.
6. Objetos almacenados en OCI Object Storage.

# 18. Checklist de entrega

- [ ] PDF/Markdown/TXT funcional.
- [ ] Chunking real.
- [ ] Embeddings.
- [ ] FAISS / búsqueda vectorial.
- [ ] RAG.
- [ ] LLM.
- [ ] Adaptación a 2+ perfiles.
- [ ] Adaptación a 2+ formatos.
- [ ] JSON estructurado.
- [ ] Revisión de fidelidad.
- [ ] Interfaz Gradio.
- [ ] OCI Object Storage configurado y probado.
- [ ] Documento original visible en el bucket.
- [ ] JSON generado visible en el bucket.
- [ ] 3 escenarios demostrados.
- [ ] Repositorio GitHub.
- [ ] README con arquitectura y guía de uso.
- [ ] Video demo en YouTube.
- [ ] Enlaces finales del proyecto.

# 19. Arquitectura para README / documentación

```text
Usuario
  │
  ▼
Gradio
  │
  ▼
Documento PDF / MD / TXT
  │
  ├──────────────► OCI Object Storage (original)
  │
  ▼
Extracción + limpieza
  │
  ▼
Chunking
  │
  ▼
Embeddings
  │
  ▼
FAISS Vector Store
  │
  ▼
Recuperación RAG
  │
  ▼
Gemini — Adaptación pedagógica
  │
  ▼
Revisor de fidelidad
  │
  ▼
Pydantic / JSON estructurado
  │
  ├──────────────► OCI Object Storage (resultado)
  │
  ▼
Interfaz + descarga JSON
```

Este diagrama puede reutilizarse en el `README.md` y en la explicación del video.